# HR Analytics — Prediction du Turnover
**Master 2 UCAO | Mahamat Haroun Ibrahim**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings, os
warnings.filterwarnings("ignore")
os.makedirs("plots", exist_ok=True)
os.makedirs("models", exist_ok=True)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score)
from xgboost import XGBClassifier
import shap, pickle

plt.style.use("seaborn-v0_8-whitegrid")
print("Bibliotheques importees")

## 1. Chargement des Donnees

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(f"Shape: {df.shape}")
print(f"Attrition rate: {(df.Attrition=="Yes").mean()*100:.1f}%")
print(f"Valeurs manquantes: {df.isnull().sum().sum()}")
df.head()

## 2. Analyse Exploratoire (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#2ecc71", "#e74c3c"]
values = df["Attrition"].value_counts()
axes[0].pie(values, labels=["Reste", "Part"], colors=colors, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Distribution Attrition", fontweight="bold")
bars = axes[1].bar(["Non", "Oui"], values.values, color=colors, edgecolor="black", alpha=0.85)
for bar, val in zip(bars, values.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, str(val), ha="center", fontweight="bold")
axes[1].set_title("Nombre par Categorie", fontweight="bold")
plt.tight_layout()
plt.savefig("plots/01_distribution_cible.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
key_vars = ["Age", "MonthlyIncome", "TotalWorkingYears", "YearsAtCompany", "DistanceFromHome", "JobSatisfaction"]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors_map = {"No": "#2ecc71", "Yes": "#e74c3c"}
for i, var in enumerate(key_vars):
    for label, grp in df.groupby("Attrition"):
        axes[i].hist(grp[var], alpha=0.6, label=label, color=colors_map[label], bins=20, edgecolor="white")
    axes[i].set_title(f"{var} par Attrition", fontweight="bold")
    axes[i].legend()
plt.suptitle("Variables Cles vs Attrition", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/02_variables_cles.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cat_vars = ["Department", "JobRole", "OverTime", "MaritalStatus", "BusinessTravel"]
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
for i, var in enumerate(cat_vars):
    ct = pd.crosstab(df[var], df["Attrition"], normalize="index") * 100
    ct.plot(kind="bar", ax=axes[i], color=["#2ecc71", "#e74c3c"], alpha=0.85, edgecolor="black")
    axes[i].set_title(f"Attrition par {var}", fontweight="bold")
    axes[i].set_ylabel("% employes")
    axes[i].tick_params(axis="x", rotation=30)
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[i].legend(["Reste", "Part"])
axes[-1].axis("off")
plt.suptitle("Taux Attrition par Variable Categorielle", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/03_variables_categorielles.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_corr = df.copy()
df_corr["Attrition_num"] = (df_corr["Attrition"]=="Yes").astype(int)
num_cols = df_corr.select_dtypes(include="number").columns.tolist()
corr_matrix = df_corr[num_cols].corr()
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn", center=0, linewidths=0.5, annot_kws={"size": 8})
plt.title("Matrice de Correlation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/04_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
corr_target = corr_matrix["Attrition_num"].drop("Attrition_num").abs().sort_values(ascending=False)
print("Top 10 correlations avec Attrition:")
print(corr_target.head(10))

## 3. Pretraitement

In [ ]:
df_model = df.copy()
cols_to_drop = ["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"]
df_model.drop(columns=cols_to_drop, inplace=True)
df_model["Attrition"] = (df_model["Attrition"]=="Yes").astype(int)
cat_cols = df_model.select_dtypes(include="object").columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])
X = df_model.drop("Attrition", axis=1)
y = df_model["Attrition"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Attrition rate train: {y_train.mean()*100:.1f}% | test: {y_test.mean()*100:.1f}%")

## 4. Modelisation XGBoost

In [ ]:
scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
xgb_model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight,
    eval_metric="logloss", random_state=42, verbosity=0)
cv_scores = cross_val_score(xgb_model, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring="roc_auc")
print(f"CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
xgb_model.fit(X_train, y_train)
print("Modele entraine!")

## 5. Evaluation

In [ ]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_proba)
print(classification_report(y_test, y_pred, target_names=["Reste", "Part"]))
print(f"AUC-ROC: {auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Reste","Part"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matrice de Confusion", fontweight="bold")
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#e74c3c", lw=2.5, label=f"XGBoost (AUC={auc:.3f})")
axes[1].plot([0,1],[0,1],"--", color="gray", label="Aleatoire")
axes[1].fill_between(fpr, tpr, alpha=0.1, color="#e74c3c")
axes[1].set_xlabel("Taux Faux Positifs"); axes[1].set_ylabel("Taux Vrais Positifs")
axes[1].set_title("Courbe ROC", fontweight="bold"); axes[1].legend()
plt.tight_layout()
plt.savefig("plots/05_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Explicabilite SHAP (XAI)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="dot", max_display=15, show=False)
plt.title("SHAP Summary Plot", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/07_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=15, show=False)
plt.title("Importance Globale SHAP", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/08_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
high_risk_idx = np.argmax(y_proba)
prob = y_proba[high_risk_idx]
shap_exp = shap.Explanation(values=shap_values[high_risk_idx], base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_idx].values, feature_names=X_test.columns.tolist())
plt.figure(figsize=(12, 8))
shap.waterfall_plot(shap_exp, max_display=12, show=False)
plt.title(f"Waterfall Plot — Employe a risque: {prob*100:.1f}%", fontweight="bold")
plt.tight_layout()
plt.savefig("plots/09_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Employe analyse — Probabilite de depart: {prob*100:.1f}%")

## 7. Analyse Ethique

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gender_att = pd.crosstab(df["Gender"], df["Attrition"], normalize="index") * 100
gender_att["Yes"].plot(kind="bar", ax=axes[0], color=["#e74c3c","#3498db"], edgecolor="black", alpha=0.85)
axes[0].set_title("Taux Attrition par Genre", fontweight="bold")
axes[0].set_ylabel("%"); axes[0].tick_params(axis="x", rotation=0)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
dept_att = pd.crosstab(df["Department"], df["Attrition"], normalize="index") * 100
dept_att["Yes"].plot(kind="bar", ax=axes[1], color=["#9b59b6","#e74c3c","#2ecc71"], edgecolor="black", alpha=0.85)
axes[1].set_title("Taux Attrition par Departement", fontweight="bold")
axes[1].set_ylabel("%"); axes[1].tick_params(axis="x", rotation=15)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.suptitle("Analyse des Biais — Equite Algorithmique", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/10_biais.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Sauvegarde et Conclusions

In [ ]:
with open("models/xgb_turnover_model.pkl", "wb") as f:
    pickle.dump({"model": xgb_model, "features": X.columns.tolist(), "auc": auc}, f)
print("Modele sauvegarde: models/xgb_turnover_model.pkl")
print()
print("=" * 50)
print("RESUME FINAL")
print("=" * 50)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"  Precision : {precision_score(y_test, y_pred)*100:.2f}%")
print(f"  Recall    : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"  F1-Score  : {f1_score(y_test, y_pred)*100:.2f}%")
print(f"  AUC-ROC   : {auc:.4f}")
print("=" * 50)